In [1]:
import pandas as pd

df = pd.read_csv("../data/Delivery_Logistics.csv")
print(df.shape)
print(df.columns.tolist())
print(df.head(10))
print(df.dtypes)

(25000, 15)
['delivery_id', 'delivery_partner', 'package_type', 'vehicle_type', 'delivery_mode', 'region', 'weather_condition', 'distance_km', 'package_weight_kg', 'delivery_time_hours', 'expected_time_hours', 'delayed', 'delivery_status', 'delivery_rating', 'delivery_cost']
   delivery_id  delivery_partner      package_type vehicle_type delivery_mode  \
0       250.99         delhivery  automobile parts         bike      same day   
1       250.99        xpressbees         cosmetics       ev van       express   
2       250.99         shadowfax         groceries        truck       two day   
3       250.99               dhl       electronics       ev van      same day   
4       250.99               dhl          clothing          van       two day   
5       250.99  amazon logistics         documents      ev bike       express   
6       250.99         delhivery         groceries      scooter      same day   
7       250.99        xpressbees     fragile items          van      same da

In [2]:
# Check if delivery_id is really constant/broken
print("Unique delivery_id values:", df["delivery_id"].nunique())
print("Sample delivery_id values:", df["delivery_id"].unique()[:5])

# Check for other missing/null issues
print(df.isnull().sum())

# Check delivery_status and delivery_partner value counts
print(df["delivery_status"].value_counts())
print(df["delivery_partner"].value_counts())

Unique delivery_id values: 24502
Sample delivery_id values: [250.99 251.   252.   253.   254.  ]
delivery_id            0
delivery_partner       0
package_type           0
vehicle_type           0
delivery_mode          0
region                 0
weather_condition      0
distance_km            0
package_weight_kg      0
delivery_time_hours    0
expected_time_hours    0
delayed                0
delivery_status        0
delivery_rating        0
delivery_cost          0
dtype: int64
delivery_status
delivered    18331
delayed       5341
failed        1328
Name: count, dtype: int64
delivery_partner
xpressbees          2826
fedex               2818
dhl                 2802
ekart               2801
blue dart           2798
delhivery           2786
shadowfax           2736
ecom express        2722
amazon logistics    2711
Name: count, dtype: int64


In [3]:
# Fix 1: Recover the real hour values from the nanosecond timestamps
df["delivery_time_hours_clean"] = pd.to_datetime(df["delivery_time_hours"]).astype("int64")
df["expected_time_hours_clean"] = pd.to_datetime(df["expected_time_hours"]).astype("int64")

# Sanity check against the 'delayed' column we already trust
df["recovered_delay_check"] = df["delivery_time_hours_clean"] > df["expected_time_hours_clean"]
print(pd.crosstab(df["delayed"], df["recovered_delay_check"]))

# Fix 2: Replace the broken delivery_id with a clean, guaranteed-unique one
df["delivery_id_clean"] = range(1, len(df) + 1)

print(df[["delivery_id", "delivery_id_clean", "delivery_time_hours", "delivery_time_hours_clean",
          "expected_time_hours_clean", "delayed"]].head(10))

recovered_delay_check  False  True 
delayed                            
no                     18331      0
yes                     1203   5466
   delivery_id  delivery_id_clean            delivery_time_hours  \
0       250.99                  1  1970-01-01 00:00:00.000000008   
1       250.99                  2  1970-01-01 00:00:00.000000002   
2       250.99                  3  1970-01-01 00:00:00.000000010   
3       250.99                  4  1970-01-01 00:00:00.000000006   
4       250.99                  5  1970-01-01 00:00:00.000000009   
5       250.99                  6  1970-01-01 00:00:00.000000004   
6       250.99                  7  1970-01-01 00:00:00.000000006   
7       250.99                  8  1970-01-01 00:00:00.000000004   
8       250.99                  9  1970-01-01 00:00:00.000000005   
9       250.99                 10  1970-01-01 00:00:00.000000003   

   delivery_time_hours_clean  expected_time_hours_clean delayed  
0                          8             

In [4]:
# Keep the original delayed/delivery_status as source of truth.
# Use recovered hours only to measure HOW delayed something was.
df["delay_hours"] = df["delivery_time_hours_clean"] - df["expected_time_hours_clean"]

# Cost efficiency metrics
df["cost_per_km"] = df["delivery_cost"] / df["distance_km"]
df["cost_per_kg"] = df["delivery_cost"] / df["package_weight_kg"]

# Final cleaned dataframe, keeping the useful columns
df_clean = df[[
    "delivery_id_clean", "delivery_partner", "package_type", "vehicle_type",
    "delivery_mode", "region", "weather_condition", "distance_km",
    "package_weight_kg", "delivery_time_hours_clean", "expected_time_hours_clean",
    "delay_hours", "delayed", "delivery_status", "delivery_rating",
    "delivery_cost", "cost_per_km", "cost_per_kg"
]].rename(columns={
    "delivery_id_clean": "delivery_id",
    "delivery_time_hours_clean": "delivery_time_hours",
    "expected_time_hours_clean": "expected_time_hours",
})

df_clean.to_csv("../data/shipments_cleaned.csv", index=False)
print("Saved. Rows:", len(df_clean))
print(df_clean.head())

Saved. Rows: 25000
   delivery_id delivery_partner      package_type vehicle_type delivery_mode  \
0            1        delhivery  automobile parts         bike      same day   
1            2       xpressbees         cosmetics       ev van       express   
2            3        shadowfax         groceries        truck       two day   
3            4              dhl       electronics       ev van      same day   
4            5              dhl          clothing          van       two day   

    region weather_condition  distance_km  package_weight_kg  \
0     west             clear        297.0              46.96   
1  central              cold         89.6              47.39   
2     east             rainy        273.5              26.89   
3     east              cold        269.7              12.69   
4    north             foggy        256.7              37.02   

   delivery_time_hours  expected_time_hours  delay_hours delayed  \
0                    8                    8    

In [5]:
import pandas as pd

# Load raw data
df = pd.read_csv("../data/Delivery_Logistics.csv")

# Recover the real hour values from the corrupted nanosecond timestamps
df["delivery_time_hours_clean"] = pd.to_datetime(df["delivery_time_hours"]).astype("int64")
df["expected_time_hours_clean"] = pd.to_datetime(df["expected_time_hours"]).astype("int64")

# Replace the broken delivery_id with a clean, guaranteed-unique one
df["delivery_id_clean"] = range(1, len(df) + 1)

# Calculate delay magnitude (trusting original 'delayed' column for yes/no status)
df["delay_hours"] = df["delivery_time_hours_clean"] - df["expected_time_hours_clean"]

# Cost efficiency metrics
df["cost_per_km"] = df["delivery_cost"] / df["distance_km"]
df["cost_per_kg"] = df["delivery_cost"] / df["package_weight_kg"]

# Final cleaned dataframe
df_clean = df[[
    "delivery_id_clean", "delivery_partner", "package_type", "vehicle_type",
    "delivery_mode", "region", "weather_condition", "distance_km",
    "package_weight_kg", "delivery_time_hours_clean", "expected_time_hours_clean",
    "delay_hours", "delayed", "delivery_status", "delivery_rating",
    "delivery_cost", "cost_per_km", "cost_per_kg"
]].rename(columns={
    "delivery_id_clean": "delivery_id",
    "delivery_time_hours_clean": "delivery_time_hours",
    "expected_time_hours_clean": "expected_time_hours",
})

df_clean.to_csv("../data/shipments_cleaned.csv", index=False)
print("Saved. Rows:", len(df_clean))
print(df_clean.head())

Saved. Rows: 25000
   delivery_id delivery_partner      package_type vehicle_type delivery_mode  \
0            1        delhivery  automobile parts         bike      same day   
1            2       xpressbees         cosmetics       ev van       express   
2            3        shadowfax         groceries        truck       two day   
3            4              dhl       electronics       ev van      same day   
4            5              dhl          clothing          van       two day   

    region weather_condition  distance_km  package_weight_kg  \
0     west             clear        297.0              46.96   
1  central              cold         89.6              47.39   
2     east             rainy        273.5              26.89   
3     east              cold        269.7              12.69   
4    north             foggy        256.7              37.02   

   delivery_time_hours  expected_time_hours  delay_hours delayed  \
0                    8                    8    